<a href="https://colab.research.google.com/github/fiqriadi25/monitoringrekon/blob/main/Analisis_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Master Baru & Generate Kode Zona

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import io

# Fungsi untuk upload file
def upload_file():
    print("Silakan upload file Excel data_master_olah.xlsx")
    uploaded = files.upload()

    for filename in uploaded.keys():
        print(f'File "{filename}" berhasil diupload ({len(uploaded[filename])} bytes)')

    if not uploaded:
        print("Tidak ada file yang diupload!")
        return None

    # Mengambil file pertama yang diupload
    file_name = list(uploaded.keys())[0]
    file_content = uploaded[file_name]

    return file_name, file_content

# Fungsi untuk membuat kode kecamatan unik (4 awal + 3 akhir)
def generate_kecamatan_code(kecamatan_name):
    kecamatan_name = str(kecamatan_name).strip().upper().replace(' ', '')

    # Jika nama kecamatan pendek (6 karakter atau kurang), gunakan langsung
    if len(kecamatan_name) <= 7:
        return kecamatan_name

    # Ambil 4 karakter pertama dan 3 karakter terakhir
    first_four = kecamatan_name[:4]
    last_three = kecamatan_name[-3:]

    kec_code = first_four + last_three

    return kec_code

# Upload file
file_name, file_content = upload_file()

if file_name:
    # Membaca data dari file Excel
    try:
        df = pd.read_excel(io.BytesIO(file_content), sheet_name='DATA MASTER')
        print(f"File berhasil dibaca. Total baris: {len(df)}")

        # Membuat salinan dataframe untuk bekerja
        df_processed = df.copy()

        # Membuat dictionary untuk kode kecamatan
        kecamatan_codes = {}

        # Generate kode untuk setiap kecamatan
        unique_kecamatan = df_processed['KECAMATAN'].unique()
        for kecamatan in unique_kecamatan:
            kec_code = generate_kecamatan_code(kecamatan)
            kecamatan_codes[kecamatan] = kec_code

        print("\nKode kecamatan yang dihasilkan (4 awal + 3 akhir):")
        for kecamatan, code in kecamatan_codes.items():
            print(f"{kecamatan} -> {code}")

        # Daftar kolom UJP yang akan digunakan
        ujp_columns = [
            'UJP CDD', 'UJP CDD-L', 'UJP CDD-C', 'UJP CDE',
            'UJP CDE-L', 'UJP CDE-C', 'UJP L300', 'UJP FUSO',
            'UJP CDD-LC', 'UJP CDE-LC', 'UJP Tronton', 'UJP Wingbox', 'UJP BlindVan'
        ]

        # Cek kolom UJP mana yang ada dalam dataframe
        available_ujp_columns = [col for col in ujp_columns if col in df_processed.columns]
        print(f"\nKolom UJP yang tersedia: {available_ujp_columns}")

        # Membuat mapping unik untuk setiap kombinasi Kecamatan + Semua UJP
        zone_mapping = {}
        kecamatan_ujp_counter = {}

        # Fungsi untuk membuat kode zona yang benar-benar unik dengan semua UJP
        def generate_unique_zone_code(row):
            # Mengambil komponen dasar
            op = str(row['OP']).strip()
            kecamatan = str(row['KECAMATAN']).strip()

            # Dapatkan kode kecamatan
            kec_code = kecamatan_codes[kecamatan]

            # Buat key unik berdasarkan kombinasi semua UJP yang tersedia
            ujp_values = []
            for ujp_col in available_ujp_columns:
                ujp_value = row[ujp_col]
                ujp_values.append(f"{ujp_col}:{ujp_value}")

            zone_key = f"{kecamatan}_" + "_".join(ujp_values)

            # Jika kombinasi ini sudah ada, gunakan kode yang sama
            if zone_key in zone_mapping:
                return zone_mapping[zone_key]

            # Jika belum ada, buat kode baru
            base_code = f"{op}{kec_code}"

            # Inisialisasi counter untuk kecamatan ini
            if kecamatan not in kecamatan_ujp_counter:
                kecamatan_ujp_counter[kecamatan] = {}

            # Buat key untuk pola semua UJP
            ujp_pattern_key = "_".join(ujp_values)

            # Jika pola UJP ini sudah ada untuk kecamatan ini, gunakan suffix yang sama
            if ujp_pattern_key in kecamatan_ujp_counter[kecamatan]:
                suffix = kecamatan_ujp_counter[kecamatan][ujp_pattern_key]
            else:
                # Tentukan suffix baru berdasarkan urutan munculnya pola UJP untuk kecamatan ini
                existing_patterns = len(kecamatan_ujp_counter[kecamatan])
                suffix = str(existing_patterns + 1)
                kecamatan_ujp_counter[kecamatan][ujp_pattern_key] = suffix

            # Buat kode zona lengkap
            zone_code = f"{base_code}{suffix}"

            # Pastikan panjang maksimal 15 karakter
            if len(zone_code) > 15:
                # Kurangi dari kode kecamatan jika perlu
                excess_chars = len(zone_code) - 15
                kec_code_shortened = kec_code[:-excess_chars]
                zone_code = f"{op}{kec_code_shortened}{suffix}"

            # Simpan mapping
            zone_mapping[zone_key] = zone_code

            return zone_code

        # Mengaplikasikan fungsi ke setiap baris
        zone_codes = []
        for index, row in df_processed.iterrows():
            zone_code = generate_unique_zone_code(row)
            zone_codes.append(zone_code)

        # Menambahkan kolom P2 ke dataframe
        df_processed['P2'] = zone_codes

        # Menampilkan beberapa baris untuk verifikasi
        print("\nContoh hasil kode zona:")
        display_columns = ['KODE TOKO', 'NAMA PENERIMA', 'OP', 'KECAMATAN', 'P2'] + available_ujp_columns
        sample_data = df_processed[display_columns].head(20)
        print(sample_data)

        # Menampilkan statistik kode zona yang dihasilkan
        print(f"\nTotal baris: {len(df_processed)}")
        print(f"Jumlah kode zona unik: {df_processed['P2'].nunique()}")

        # Menampilkan contoh kode zona untuk kecamatan tertentu dengan UJP berbeda
        print("\nContoh kode zona untuk beberapa kecamatan (dengan pola UJP berbeda):")
        sample_kecamatan = ['Sungai Raya', 'Mempawah Hilir', 'Mempawah Hulu', 'Pontianak Barat']

        for kec in sample_kecamatan:
            if kec in df_processed['KECAMATAN'].values:
                kec_data = df_processed[df_processed['KECAMATAN'] == kec][['KECAMATAN', 'P2'] + available_ujp_columns].drop_duplicates()
                print(f"\n{kec} - {len(kec_data)} pola UJP unik:")

                for idx, row in kec_data.iterrows():
                    ujp_info = []
                    for ujp_col in available_ujp_columns:
                        if pd.notna(row[ujp_col]) and row[ujp_col] != 0:
                            ujp_info.append(f"{ujp_col}: {row[ujp_col]}")

                    print(f"  {row['P2']} -> {', '.join(ujp_info)}")

        # Verifikasi khusus untuk beberapa kecamatan
        print("\n=== VERIFIKASI POLA UJP UNIK ===")
        for kec in sample_kecamatan[:2]:  # Cek 2 kecamatan pertama
            if kec in df_processed['KECAMATAN'].values:
                kec_data = df_processed[df_processed['KECAMATAN'] == kec][['KECAMATAN', 'P2'] + available_ujp_columns].drop_duplicates()
                print(f"\n{kec} - Pola UJP Unik:")
                print(kec_data[['P2'] + available_ujp_columns].head(10))

        # Cek duplikasi
        print(f"\nPengecekan duplikasi:")
        check_columns = ['P2', 'KECAMATAN'] + available_ujp_columns
        duplicate_check = df_processed[check_columns].drop_duplicates()
        duplicate_zones = duplicate_check[duplicate_check.duplicated('P2', keep=False)]

        if len(duplicate_zones) > 0:
            print("⚠️  Ditemukan duplikasi kode zona:")
            print(duplicate_zones)
        else:
            print("✅ Tidak ditemukan duplikasi kode zona")

        # Cek panjang kode zona
        print(f"\nPengecekan panjang kode zona:")
        df_processed['P2_LENGTH'] = df_processed['P2'].str.len()
        length_stats = df_processed['P2_LENGTH'].describe()
        print(length_stats)

        # Tampilkan kode zona terpanjang
        longest_codes = df_processed.nlargest(5, 'P2_LENGTH')[['KECAMATAN', 'P2', 'P2_LENGTH']]
        print("\n5 Kode zona terpanjang:")
        print(longest_codes)

        # Menyimpan hasil ke file Excel baru (tanpa kolom length)
        df_processed.drop('P2_LENGTH', axis=1, inplace=True)
        output_file = 'data_master_olah_BYL.xlsx'
        df_processed.to_excel(output_file, index=False)
        print(f"\nFile hasil telah disimpan sebagai: {output_file}")

        # Download file hasil
        files.download(output_file)
        print("File hasil telah didownload!")

        # Summary akhir
        print(f"\n=== SUMMARY AKHIR ===")
        print(f"Total baris data: {len(df_processed)}")
        print(f"Total kode zona unik: {df_processed['P2'].nunique()}")
        print(f"Kolom UJP yang digunakan: {len(available_ujp_columns)}")
        print(f"Kecamatan unik: {len(unique_kecamatan)}")

    except Exception as e:
        print(f"Error dalam membaca file: {e}")
        import traceback
        print(traceback.format_exc())

else:
    print("Upload file dibatalkan.")

# Penambahan Toko & Generate Kode Zona

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import io
import base64
import re

def process_zone_codes():
    # Upload file
    print("📤 Silakan upload file Excel data_master.xlsx")
    uploaded = files.upload()

    # Cek file yang diupload
    file_name = None
    for name in uploaded.keys():
        if name.endswith('.xlsx'):
            file_name = name
            break

    if not file_name:
        print("❌ File Excel tidak ditemukan. Pastikan file berekstensi .xlsx")
        return

    print(f"✅ File '{file_name}' berhasil diupload!")

    # Baca file Excel
    try:
        data_master_df = pd.read_excel(io.BytesIO(uploaded[file_name]), sheet_name='DATA MASTER')
        master_toko_df = pd.read_excel(io.BytesIO(uploaded[file_name]), sheet_name='MASTER TOKO')
    except Exception as e:
        print(f"❌ Error membaca file: {e}")
        return

    # Robust column cleaning function
    def clean_col_names(df):
        cols = df.columns
        new_cols = []
        for col in cols:
            # Convert to lowercase
            new_col = str(col).lower()
            # Replace non-alphanumeric characters with underscores
            new_col = re.sub(r'\W+', '_', new_col)
            # Remove leading/trailing underscores
            new_col = new_col.strip('_')
            new_cols.append(new_col)
        df.columns = new_cols
        return df

    # Apply robust column cleaning
    data_master_df = clean_col_names(data_master_df)
    master_toko_df = clean_col_names(master_toko_df)

    print("Debug: master_toko_df columns after cleaning:")
    print(master_toko_df.columns)
    print("Debug: data_master_df columns after cleaning:")
    print(data_master_df.columns)

    # Fungsi untuk mendapatkan nilai UJP sebagai tuple untuk perbandingan
    def get_ujp_tuple(row):
        ujp_columns = [
            'ujp_cdd', 'ujp_cdd_l', 'ujp_cdd_c', 'ujp_cde',
            'ujp_cde_l', 'ujp_cde_c', 'ujp_l300', 'ujp_fuso',
            'ujp_cdd_lc', 'ujp_cde_lc', 'ujp_tronton', 'ujp_wingbox', 'ujp_blindvan'
        ]
        return tuple(row[col] for col in ujp_columns if col in row)

    # Fungsi untuk menghasilkan kode zona dengan batasan 12-15 karakter
    def generate_zone_code(op, kecamatan, suffix):
        # Bersihkan nama kecamatan (hilangkan spasi, uppercase)
        kecamatan_clean = kecamatan.upper().replace(' ', '')

        # Gabungkan OP + Kecamatan + Suffix
        suffix_str = str(suffix)
        base_combination = f"{op}{kecamatan_clean}"

        # Hitung panjang tanpa suffix
        base_length = len(base_combination)
        total_length = base_length + len(suffix_str)

        # Jika total length <= 15, gunakan kombinasi lengkap
        if total_length <= 15:
            final_code = f"{base_combination}{suffix_str}"
        else:
            # Jika terlalu panjang, gunakan logika 4 awal + 3 akhir
            first_part = kecamatan_clean[:4] if len(kecamatan_clean) >= 4 else kecamatan_clean.ljust(4, 'X')
            last_part = kecamatan_clean[-3:] if len(kecamatan_clean) >= 3 else kecamatan_clean.rjust(3, 'X')
            kec_code = f"{first_part}{last_part}"
            kec_code = ''.join(e for e in kec_code if e.isalnum())

            base_combination = f"{op}{kec_code}"
            final_code = f"{base_combination}{suffix_str}"

        # Validasi panjang akhir (12-15 karakter)
        final_length = len(final_code)

        if final_length < 12:
            # Tambahkan padding jika kurang dari 12 karakter
            needed_padding = 12 - final_length
            padding = kecamatan_clean[:needed_padding] if len(kecamatan_clean) >= needed_padding else kecamatan_clean.ljust(needed_padding, 'X')
            final_code = f"{op}{kecamatan_clean}{padding}{suffix_str}"[:12]
        elif final_length > 15:
            # Potong jika lebih dari 15 karakter
            max_base_length = 15 - len(suffix_str)
            final_code = f"{base_combination[:max_base_length]}{suffix_str}"

        return final_code

    # Process DATA MASTER
    print("🔄 Memproses DATA MASTER...")

    # Hitung baris dengan Kode Zona kosong sebelum diproses
    empty_zones_before = data_master_df['kode_zona'].isna() | (data_master_df['kode_zona'] == '')
    print(f"📊 Jumlah Kode Zona kosong sebelum diproses: {empty_zones_before.sum()}")

    # Buat dictionary dari MASTER TOKO untuk referensi
    toko_zone_mapping = {}
    for _, row in master_toko_df.iterrows():
        # Use the standardized column names
        kecamatan = row['kecamatan']
        op = row['operating_point']

        # Dapatkan nilai UJP
        ujp_tuple = get_ujp_tuple(row)

        # Gabungkan kunci: (OP, Kecamatan, UJP tuple)
        key = (op, kecamatan, ujp_tuple)
        kode_zona = row['kode_zona']

        if key not in toko_zone_mapping:
            toko_zone_mapping[key] = kode_zona

    # Kelompokkan data berdasarkan OP and Kecamatan to determine suffix
    zone_groups = {}
    for _, row in data_master_df.iterrows():
        if pd.isna(row['kode_zona']) or row['kode_zona'] == '':
            op = row['operating_point'] # Use standardized column name
            kecamatan = row['kecamatan']
            ujp_tuple = get_ujp_tuple(row)

            group_key = (op, kecamatan)
            if group_key not in zone_groups:
                zone_groups[group_key] = []

            zone_groups[group_key].append((ujp_tuple, row.name))

    # For each group, determine suffix based on UJP pattern
    zone_suffix_mapping = {}
    for group_key, ujp_records in zone_groups.items():
        op, kecamatan = group_key

        # Kelompokkan berdasarkan pola UJP
        ujp_patterns = {}
        suffix_counter = 1

        for ujp_tuple, idx in ujp_records:
            if ujp_tuple not in ujp_patterns:
                # Check if this UJP pattern exists in MASTER TOKO
                found_in_toko = False
                for toko_key, toko_zone in toko_zone_mapping.items():
                    if toko_key[0] == op and toko_key[1] == kecamatan and toko_key[2] == ujp_tuple:
                        zone_suffix_mapping[idx] = toko_zone
                        found_in_toko = True
                        final_code = generate_zone_code(op, kecamatan, suffix_counter)
                        print(f"✅ Kode zona from MASTER TOKO: {toko_zone} for {op} - {kecamatan} (Panjang: {len(toko_zone)})")
                        break

                if not found_in_toko:
                    ujp_patterns[ujp_tuple] = suffix_counter
                    new_zone_code = generate_zone_code(op, kecamatan, suffix_counter)
                    zone_suffix_mapping[idx] = new_zone_code

                    # Tampilkan informasi pembuatan kode
                    kecamatan_clean = kecamatan.upper().replace(' ', '')
                    base_combination = f"{op}{kecamatan_clean}"
                    total_with_suffix = len(base_combination) + len(str(suffix_counter))

                    if total_with_suffix <= 15:
                        method = "FULL"
                    else:
                        method = "SHORT"

                    print(f"🆕 New zone code: {new_zone_code} for {op} - {kecamatan} (Method: {method}, Pattern {suffix_counter}, Panjang: {len(new_zone_code)})")
                    suffix_counter += 1
            else:
                existing_suffix = ujp_patterns[ujp_tuple]
                zone_suffix_mapping[idx] = generate_zone_code(op, kecamatan, existing_suffix)

    # Apply zone codes to DATA MASTER
    for idx, kode_zona in zone_suffix_mapping.items():
        data_master_df.at[idx, 'kode_zona'] = kode_zona

    # Count rows with empty Zone Code after processing
    empty_zones_after = data_master_df['kode_zona'].isna() | (data_master_df['kode_zona'] == '')
    print(f"📊 Number of empty Zone Codes after processing: {empty_zones_after.sum()}")

    # Tampilkan contoh kode zona yang dihasilkan
    print("\n🔍 Contoh Kode Zona yang Dihasilkan:")
    sample_kecamatan = data_master_df['kecamatan'].unique()[:5]
    for kec in sample_kecamatan:
        sample_zone_code_1 = generate_zone_code("LSATPNKF", kec, 1)
        kecamatan_clean = kec.upper().replace(' ', '')
        base_combination = f"LSATPNKF{kecamatan_clean}"
        total_with_suffix = len(base_combination) + 1

        method = "FULL" if total_with_suffix <= 15 else "SHORT"
        print(f"  {kec} → {sample_zone_code_1} (Method: {method}, Panjang: {len(sample_zone_code_1)})")

    # Validasi panjang kode zona
    print("\n📏 Validasi Panjang Kode Zona:")
    zone_lengths = data_master_df['kode_zona'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)
    print(f"  Panjang minimum: {zone_lengths.min()} karakter")
    print(f"  Panjang maksimum: {zone_lengths.max()} karakter")
    print(f"  Rata-rata panjang: {zone_lengths.mean():.1f} karakter")

    # Cek jika ada yang tidak memenuhi syarat
    invalid_lengths = data_master_df[(zone_lengths < 12) | (zone_lengths > 15)]
    if len(invalid_lengths) > 0:
        print(f"  ⚠️  Peringatan: {len(invalid_lengths)} kode zona tidak memenuhi batasan 12-15 karakter")
        print("  Kode zona yang bermasalah:")
        for idx, row in invalid_lengths.iterrows():
            print(f"    {row['kode_toko']}: {row['kode_zona']} (Panjang: {len(str(row['kode_zona']))})")
    else:
        print("  ✅ Semua kode zona memenuhi batasan 12-15 karakter")

    # Save results to Excel file in memory
    output = io.BytesIO()
    output_filename = f'data_master_new{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
    with pd.ExcelWriter(output, engine='openpyxl') as writer:
        data_master_df.to_excel(writer, sheet_name='DATA MASTER', index=False)
        master_toko_df.to_excel(writer, sheet_name='MASTER TOKO', index=False)

    output.seek(0)

    # Save the file to the Colab environment
    with open(output_filename, 'wb') as f:
        f.write(output.getvalue())

    # Download result file
    print("\n📥 Downloading result file...")
    files.download(output_filename)

    # Display preview of results
    print("\n📋 Preview of DATA MASTER results:")
    preview_cols = ['kode_toko', 'nama_penerima', 'kecamatan', 'kode_zona']
    available_cols = [col for col in preview_cols if col in data_master_df.columns]
    preview_df = data_master_df[available_cols].head(10).copy()
    preview_df['panjang'] = preview_df['kode_zona'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)
    print(preview_df.to_string(index=False))

    print("\n✅ Process completed! Result file has been downloaded.")

# Run the main function
if __name__ == "__main__":
    process_zone_codes()

# Update UJP Master

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import io
import re

def process_ujp_update():
    print("📤 Silakan upload file Excel 'Master_Toko.xlsx'")
    uploaded = files.upload()

    file_name = None
    for name in uploaded.keys():
        if name.endswith('.xlsx'):
            file_name = name
            break

    if not file_name:
        print("❌ File Excel tidak ditemukan. Pastikan file berekstensi .xlsx")
        return

    print(f"✅ File '{file_name}' berhasil diupload!")

    try:
        data_master_df = pd.read_excel(io.BytesIO(uploaded[file_name]), sheet_name='DATA MASTER')
        print("✅ Berhasil membaca sheet DATA MASTER")
    except Exception as e:
        print(f"❌ Error membaca file: {e}")
        return

    def clean_col_names(df):
        cols = df.columns
        new_cols = []
        for col in cols:
            new_col = str(col).lower()
            new_col = re.sub(r'\W+', '_', new_col)
            new_col = new_col.strip('_')
            new_cols.append(new_col)
        df.columns = new_cols
        return df

    data_master_df = clean_col_names(data_master_df)

    ujp_columns_original = [
        'ujp_cdd', 'ujp_cdd_l', 'ujp_cdd_c', 'ujp_cde',
        'ujp_cde_l', 'ujp_cde_c', 'ujp_l300', 'ujp_fuso',
        'ujp_cdd_lc', 'ujp_cde_lc', 'ujp_tronton', 'ujp_wingbox',
        'ujp_blindvan', 'ujp_l300_c'
    ]

    ujp_columns_update = [
        'update_ujp_cdd', 'update_ujp_cdd_l', 'update_ujp_cdd_c',
        'update_ujp_cde', 'update_ujp_cde_l', 'update_ujp_cde_c',
        'update_ujp_l300', 'update_ujp_fuso', 'update_ujp_cdd_lc',
        'update_ujp_cde_lc', 'update_ujp_tronton', 'update_ujp_wingbox',
        'update_ujp_blindvan', 'update_ujp_l300_c'
    ]

    def get_ujp_tuple(row, columns):
        return tuple(row[col] if col in row and pd.notna(row[col]) else 0 for col in columns)

    def generate_zone_code(op, kecamatan, suffix):
        base_code = f"{op}{kecamatan.upper()}"
        base_code = ''.join(e for e in base_code if e.isalnum())[:15-len(str(suffix))]
        return f"{base_code}{suffix}"

    def find_matching_zone(op, kecamatan, ujp_tuple, exclude_index=None):
        matching_zones = []
        for idx, row in data_master_df.iterrows():
            if exclude_index is not None and idx == exclude_index:
                continue
            if (row['operating_point'] == op and
                row['kecamatan'] == kecamatan and
                get_ujp_tuple(row, ujp_columns_update) == ujp_tuple):
                matching_zones.append((idx, row['kode_zona']))
        return matching_zones

    def get_existing_suffixes(op, kecamatan):
        existing_suffixes = set()
        pattern = r'^' + re.escape(op) + re.escape(kecamatan.upper()) + r'(\d+)$'
        for idx, row in data_master_df.iterrows():
            current_zone = row['kode_zona']
            if (isinstance(current_zone, str) and
                row['operating_point'] == op and
                row['kecamatan'] == kecamatan):
                match = re.match(pattern, current_zone)
                if match:
                    suffix = int(match.group(1))
                    existing_suffixes.add(suffix)
        return sorted(existing_suffixes)

    def get_new_suffix(op, kecamatan):
        existing_suffixes = get_existing_suffixes(op, kecamatan)
        if not existing_suffixes:
            return 1
        max_suffix = max(existing_suffixes)
        for i in range(1, max_suffix + 2):
            if i not in existing_suffixes:
                return i
        return max_suffix + 1

    print("🔄 Memproses update UJP dan Kode Zona...")

    results = []

    for idx, row in data_master_df.iterrows():
        op = row['operating_point']
        kecamatan = row['kecamatan']
        current_zone = row['kode_zona']

        ujp_original = get_ujp_tuple(row, ujp_columns_original)
        ujp_updated = get_ujp_tuple(row, ujp_columns_update)

        keterangan = ""
        new_zone = current_zone

        # VALIDASI BARU
        if ujp_original == ujp_updated:
            keterangan = "HANYA UPDATE UJP !"
            new_zone = current_zone
        else:
            matching_zones = find_matching_zone(op, kecamatan, ujp_updated, exclude_index=idx)

            if matching_zones:
                existing_zone = matching_zones[0][1]
                if existing_zone != current_zone:
                    new_zone = existing_zone
                    keterangan = "UPDATE UJP & PINDAH ZONA"
                else:
                    new_zone = current_zone
                    keterangan = "PINDAH ZONA"
            else:
                new_suffix = get_new_suffix(op, kecamatan)
                new_zone = generate_zone_code(op, kecamatan, new_suffix)
                keterangan = "UPDATE UJP & ZONA BARU"
                print(f"🆕 Generated new zone: {new_zone} for {row['kode_toko']} (suffix: {new_suffix})")

        results.append({
            'index': idx,
            'kode_toko': row['kode_toko'],
            'kecamatan': kecamatan,
            'zone_awal': current_zone,
            'zone_baru': new_zone,
            'keterangan': keterangan
        })

        data_master_df.at[idx, 'kode_zona'] = new_zone
        data_master_df.at[idx, 'keterangan'] = keterangan

    output = io.BytesIO()
    output_filename = f'Updated_UJP_Master_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
    with pd.ExcelWriter(output, engine='openpyxl') as writer:
        data_master_df.to_excel(writer, sheet_name='DATA MASTER', index=False)

    output.seek(0)
    with open(output_filename, 'wb') as f:
        f.write(output.getvalue())

    print("\n📥 Download file hasil...")
    files.download(output_filename)

    print("\n📊 STATISTIK PERUBAHAN:")
    keterangan_counts = pd.Series([r['keterangan'] for r in results]).value_counts()
    for keterangan, count in keterangan_counts.items():
        print(f"{keterangan}: {count} toko")

    print(f"\n✅ Total toko diproses: {len(results)}")
    print("✅ Proses selesai! File hasil telah didownload.")

if __name__ == "__main__":
    process_ujp_update()

# Master Upload Driver

In [ ]:
from google.colab import files
import pandas as pd
import openpyxl

# Upload file
print("📤 Upload file Excel master_driver_gform.xlsx")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print(f"✅ File '{file_name}' berhasil diupload!")

# Baca file Excel
wb = openpyxl.load_workbook(file_name)

# PROSES DATA PENGAJUAN - MENGELOMPOKKAN BEBERAPA DRIVER
pengajuan_sheet = wb['DATA PENGAJUAN']
pengajuan_data = []

print("\n🔍 Membaca data dari sheet DATA PENGAJUAN...")

# Kumpulkan semua baris data
all_rows = []
for row in pengajuan_sheet.iter_rows(values_only=True):
    if row and row[0]:  # Hanya baris yang tidak kosong
        all_rows.append(row[0])

# Kelompokkan data per driver
drivers_data = []
current_driver = {}

for item in all_rows:
    if item and ":" in str(item):
        key, value = str(item).split(":", 1)
        key = key.strip()
        value = value.strip()

        # Jika menemukan NAMA LENGKAP DRIVER baru, simpan driver sebelumnya dan mulai baru
        if key == "NAMA LENGKAP DRIVER" and current_driver:
            drivers_data.append(current_driver)
            current_driver = {}

        current_driver[key] = value

# Tambahkan driver terakhir
if current_driver:
    drivers_data.append(current_driver)

print(f"📊 Ditemukan {len(drivers_data)} driver dalam DATA PENGAJUAN")

# PROSES MASTER DRIVER
master_sheet = wb['MASTER DRIVER']

# Ambil header MASTER DRIVER
headers = [cell.value for cell in master_sheet[1]]
print(f"\n📝 Header MASTER DRIVER: {headers}")

# Ambil NIK yang sudah ada di MASTER DRIVER untuk pengecekan duplikat
existing_niks = []
for row in master_sheet.iter_rows(min_row=2, values_only=True):
    if row[7]:  # Kolom NIK (index 7)
        existing_niks.append(str(row[7]).strip())

print(f"📋 Jumlah driver yang sudah ada di MASTER DRIVER: {len(existing_niks)}")

# Mapping antara DATA PENGAJUAN dan MASTER DRIVER
mapping = {
    "NAMA LENGKAP DRIVER": "NAMA DRIVER",
    "NIK DRIVER": "NIK",
    "VENDOR DRIVER": "NAMA VENDOR",
    "TEMPAT LAHIR": "TEMPAT LAHIR",
    "TANGGAL LAHIR": "TANGGAL LAHIR",
    "ALAMAT LENGKAP SESUAI KTP": "ALAMAT",
    "NO HANDPHONE": "NO HP",
    "LISENSI SIM": "SIM",
    "NO SIM": "ID DRIVER",
    "TANGGAL VALIDITAS SIM": "SIM VALID"
}

# PROSES SETIAP DRIVER
added_count = 0
skipped_count = 0

for i, driver_data in enumerate(drivers_data, 1):
    print(f"\n{'='*50}")
    print(f"Memproses Driver {i}: {driver_data.get('NAMA LENGKAP DRIVER', 'N/A')}")
    print(f"{'='*50}")

    # Mapping data ke format MASTER DRIVER
    master_driver = {}
    for pengajuan_key, master_key in mapping.items():
        master_driver[master_key] = driver_data.get(pengajuan_key, "")

    # Tambahkan kolom default
    master_driver["JABATAN"] = "DRIVER"
    master_driver["OP"] = ""
    master_driver["KODE VENDOR"] = ""

    # Cek duplikat berdasarkan NIK
    driver_nik = master_driver.get("NIK", "").strip()

    if driver_nik in existing_niks:
        print(f"❌ SKIP: Driver {driver_data.get('NAMA LENGKAP DRIVER', 'N/A')} dengan NIK {driver_nik} sudah ada")
        skipped_count += 1
        continue

    # Siapkan data dalam urutan header
    new_row = []
    for header in headers:
        new_row.append(master_driver.get(header, ""))

    # Tambahkan baris baru ke MASTER DRIVER
    next_row = master_sheet.max_row + 1
    for col_idx, value in enumerate(new_row, 1):
        master_sheet.cell(row=next_row, column=col_idx, value=value)

    # Tambahkan NIK ke daftar existing untuk menghindari duplikat dalam sesi yang sama
    existing_niks.append(driver_nik)
    added_count += 1

    print(f"✅ BERHASIL: Data {driver_data.get('NAMA LENGKAP DRIVER', 'N/A')} ditambahkan")

    # Tampilkan detail driver yang ditambahkan
    for key, value in master_driver.items():
        if value:  # Hanya tampilkan yang ada nilai
            print(f"   {key}: {value}")

# Simpan file
if added_count > 0:
    wb.save(file_name)
    print(f"\n🎉 PROSES SELESAI!")
    print(f"✅ {added_count} driver berhasil ditambahkan")
    print(f"⏭️  {skipped_count} driver dilewati (sudah ada)")
else:
    print(f"\nℹ️  Tidak ada data baru yang ditambahkan")

# Preview hasil akhir
print(f"\n📊 Preview data terakhir di MASTER DRIVER:")
master_df = pd.read_excel(file_name, sheet_name='MASTER DRIVER')
print(f"Total driver di MASTER DRIVER: {len(master_df)}")
print(master_df.tail(min(5, len(master_df))))

# Download file updated
if added_count > 0:
    print(f"\n📥 Download file '{file_name}' yang sudah diupdate...")
    files.download(file_name)
else:
    print("\n📁 File tidak didownload karena tidak ada perubahan")

# Geo Code

In [ ]:
# Install library yang diperlukan
!pip install pandas openpyxl requests tqdm

import pandas as pd
import requests
import time
from google.colab import files
import io
from datetime import datetime
from tqdm.notebook import tqdm

def get_address_nominatim(lat, lon):
    """
    Mendapatkan detail alamat menggunakan OpenStreetMap Nominatim (GRATIS)
    """
    try:
        url = f"https://nominatim.openstreetmap.org/reverse?format=json&lat={lat}&lon={lon}&zoom=18&addressdetails=1"
        headers = {
            'User-Agent': 'Vehicle-Tracker-App/1.0 (contact@example.com)',
            'Accept-Language': 'id'
        }

        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            data = response.json()

            if 'error' not in data:
                address = data.get('address', {})

                # Ekstrak komponen alamat dalam Bahasa Indonesia
                kecamatan = address.get('suburb') or address.get('village') or address.get('county') or '-'
                kota = address.get('city') or address.get('town') or address.get('municipality') or '-'
                provinsi = address.get('state') or address.get('region') or '-'

                return f"{kecamatan}, {kota}, {provinsi}"
            else:
                return "Data tidak ditemukan"
        else:
            return f"Error: HTTP {response.status_code}"

    except Exception as e:
        return f"Error: {str(e)}"

def get_address_geocode_xyz(lat, lon):
    """
    Alternatif menggunakan geocode.xyz (GRATIS)
    """
    try:
        url = f"https://geocode.xyz/{lat},{lon}?json=1&lang=id"
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()

            if 'city' in data and 'region' in data:
                kota = data.get('city', '-')
                provinsi = data.get('region', '-')
                kecamatan = data.get('county', '-') or data.get('subregion', '-')

                return f"{kecamatan}, {kota}, {provinsi}"
            else:
                return "Data tidak ditemukan"
        else:
            return f"Error: HTTP {response.status_code}"

    except Exception as e:
        return f"Error: {str(e)}"

print("🌍 GEOCODING VEHICLE DATA - OPENSTREETMAP (GRATIS)")
print("=" * 60)
print("Menggunakan OpenStreetMap Nominatim - TANPA API KEY")
print("=" * 60)

# STEP 1: Upload File
print("\n📁 STEP 1: Upload File Excel")
print("Pastikan file memiliki kolom: latitude dan longitude")
uploaded = files.upload()

if not uploaded:
    print("❌ Tidak ada file yang diupload!")
    exit()

file_name = list(uploaded.keys())[0]
print(f"✅ File '{file_name}' berhasil diupload!")

# Load data
try:
    df = pd.read_excel(io.BytesIO(uploaded[file_name]), sheet_name="All Vehicle")
    print(f"📊 Data loaded: {len(df)} rows, {len(df.columns)} columns")

    # Cek kolom yang diperlukan
    required_cols = ['latitude', 'longitude']
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        print(f"❌ Kolom yang tidak ditemukan: {missing_cols}")
        print(f"📋 Kolom yang tersedia: {list(df.columns)}")
        exit()

except Exception as e:
    print(f"❌ Error loading file: {e}")
    exit()

# Preview data
print("\n📋 Preview Data (5 baris pertama):")
print(df[['license_plate', 'latitude', 'longitude']].head())

# STEP 2: Pilih Provider Geocoding
print("\n🔧 STEP 2: Pilih Provider Geocoding")
print("1. OpenStreetMap Nominatim (Recommended)")
print("   - Akurat untuk Indonesia")
print("   - Rate limit: 1 request/detik")
print("2. Geocode.xyz (Alternatif)")
print("   - Global coverage")
print("   - Rate limit lebih longgar")

provider_choice = input("Pilih provider (1/2) [default: 1]: ").strip() or "1"

if provider_choice == "1":
    geocoding_func = get_address_nominatim
    provider_name = "OpenStreetMap"
    delay_time = 1.0  # 1 request per second
else:
    geocoding_func = get_address_geocode_xyz
    provider_name = "Geocode.xyz"
    delay_time = 0.5  # 2 requests per second

print(f"✅ Menggunakan provider: {provider_name}")

# STEP 3: Konfirmasi Processing
print(f"\n🚀 STEP 3: Konfirmasi Processing")
print(f"• Provider: {provider_name}")
print(f"• Total data: {len(df)} baris")
print(f"• Estimasi waktu: {len(df) * delay_time / 60:.1f} menit")
print(f"• Biaya: GRATIS 100%")

confirm = input("\nLanjutkan proses geocoding? (y/n): ").lower()
if confirm != 'y':
    print("❌ Proses dibatalkan!")
    exit()

# STEP 4: Processing dengan Progress Bar
print(f"\n⏳ STEP 4: Memproses Geocoding dengan {provider_name}...")
print("⚠️  Mohon tunggu, proses mungkin memakan waktu beberapa menit...")

result_df = df.copy()
success_count = 0
error_count = 0
results_log = []

# Process dengan progress bar
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    license_plate = row['license_plate']
    lat = row['latitude']
    lon = row['longitude']

    if pd.notna(lat) and pd.notna(lon):
        nama_daerah = geocoding_func(lat, lon)
        result_df.at[index, 'Nama Daerah'] = nama_daerah

        # Log hasil
        results_log.append({
            'No': index + 1,
            'License Plate': license_plate,
            'Latitude': lat,
            'Longitude': lon,
            'Nama Daerah': nama_daerah,
            'Status': 'SUCCESS' if 'Error' not in nama_daerah and 'tidak ditemukan' not in nama_daerah else 'ERROR'
        })

        if 'Error' not in nama_daerah and 'tidak ditemukan' not in nama_daerah:
            success_count += 1
        else:
            error_count += 1
    else:
        error_msg = "ERROR: Koordinat tidak valid"
        result_df.at[index, 'Nama Daerah'] = error_msg
        error_count += 1

        results_log.append({
            'No': index + 1,
            'License Plate': license_plate,
            'Latitude': lat,
            'Longitude': lon,
            'Nama Daerah': error_msg,
            'Status': 'ERROR'
        })

    # Rate limiting untuk menghormati provider
    time.sleep(delay_time)

# STEP 5: Results & Analysis
print("\n✅ PROSES SELESAI!")
print("=" * 50)
print(f"📊 HASIL AKHIR:")
print(f"   ✅ Berhasil: {success_count} data")
print(f"   ❌ Gagal: {error_count} data")
print(f"   📊 Total: {len(df)} data")
print(f"   🎯 Success Rate: {(success_count/len(df))*100:.1f}%")

# Convert log to DataFrame untuk analisis
log_df = pd.DataFrame(results_log)

# Tampilkan preview hasil
print("\n📋 Preview Hasil (10 data pertama):")
preview_df = log_df[['No', 'License Plate', 'Nama Daerah', 'Status']].head(10)
print(preview_df.to_string(index=False))

# Analisis detail
print(f"\n📈 ANALISIS DETAIL:")
success_data = log_df[log_df['Status'] == 'SUCCESS']
error_data = log_df[log_df['Status'] == 'ERROR']

print(f"• Data berhasil: {len(success_data)}")
print(f"• Data error: {len(error_data)}")

if len(error_data) > 0:
    print(f"\n⚠️  DETAIL ERROR:")
    error_reasons = error_data['Nama Daerah'].value_counts()
    for reason, count in error_reasons.items():
        print(f"   • {reason}: {count} data")

    # Tampilkan 5 data error pertama
    print(f"\n🔍 Sample Data Error:")
    print(error_data[['No', 'License Plate', 'Latitude', 'Longitude', 'Nama Daerah']].head(5).to_string(index=False))

# STEP 6: Download Results
print("\n📥 STEP 6: Download Hasil")

# Buat file hasil lengkap
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"Hasil_Geocoding_GRATIS_{timestamp}.xlsx"

# Simpan ke Excel dengan multiple sheets
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    # Sheet 1: Data lengkap
    result_df.to_excel(writer, sheet_name='Data Lengkap', index=False)

    # Sheet 2: Hasil berhasil
    success_data.to_excel(writer, sheet_name='Data Berhasil', index=False)

    # Sheet 3: Data error
    if len(error_data) > 0:
        error_data.to_excel(writer, sheet_name='Data Error', index=False)

    # Sheet 4: Summary
    summary_data = {
        'Metric': ['Total Data', 'Data Berhasil', 'Data Error', 'Success Rate', 'Provider', 'Tanggal Proses'],
        'Value': [
            len(df),
            success_count,
            error_count,
            f"{(success_count/len(df))*100:.1f}%",
            provider_name,
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

print(f"💾 File disimpan: {output_filename}")
print("📁 Sheets dalam file:")
print("   • Data Lengkap: Semua data dengan hasil geocoding")
print("   • Data Berhasil: Hanya data yang berhasil")
print("   • Data Error: Data yang mengalami error")
print("   • Summary: Ringkasan hasil processing")

# Download file
files.download(output_filename)

print("\n🎉 SELESAI! File hasil telah didownload.")
print("\n📝 CATATAN PENTING:")
print("• Data menggunakan OpenStreetMap - GRATIS 100%")
print("• Untuk hasil terbaik, pastikan koordinat valid")
print("• Beberapa lokasi terpencil mungkin tidak terdeteksi")
print("• File hasil berisi multiple sheets untuk analisis")

# Tampilkan sample data berhasil
if len(success_data) > 0:
    print(f"\n📍 CONTOH DATA BERHASIL:")
    sample_success = success_data[['License Plate', 'Nama Daerah']].head(3)
    for _, row in sample_success.iterrows():
        print(f"   • {row['License Plate']}: {row['Nama Daerah']}")

# Rute Zona

In [ ]:
import pandas as pd
import numpy as np

# Membaca data dari file Excel
# Note: Dalam Google Colab, upload file terlebih dahulu
from google.colab import files
uploaded = files.upload()

# Memuat file Excel
file_name = list(uploaded.keys())[0]
excel_file = pd.ExcelFile(file_name)

# Membaca sheet AOP dan data_master
df_aop = pd.read_excel(excel_file, sheet_name='AOP')
df_master = pd.read_excel(excel_file, sheet_name='data_master')

# Membuat dictionary mapping kode toko ke kota dan UJP dari data_master
kode_to_data = {}
for _, row in df_master.iterrows():
    kode = row['Kode Toko']
    kota = row['Kota']
    # Mengambil semua kolom UJP (UJP CDE, UJP CDE-L, UJP CDE-C, UJP CDE-LC)
    ujp_values = []

    # Cek kolom UJP yang ada
    ujp_columns = ['UJP CDE', 'UJP CDE-L', 'UJP CDE-C', 'UJP CDE-LC']
    for col in ujp_columns:
        if col in df_master.columns:
            ujp_val = row[col]
            # Pastikan nilai UJP adalah numerik
            if pd.notna(ujp_val) and str(ujp_val).replace('.', '', 1).isdigit():
                ujp_values.append(float(ujp_val))

    # Ambil nilai UJP tertinggi untuk toko ini
    ujp_max = max(ujp_values) if ujp_values else 0

    kode_to_data[kode] = {
        'kota': kota,
        'ujp_max': ujp_max
    }

# Fungsi untuk mendapatkan kota dengan total UJP tertinggi dari list store
def get_kota_rute_by_ujp(row):
    kota_ujp_total = {}

    # Iterasi melalui kolom Store 1 hingga Store 50
    for i in range(1, 51):
        col_name = f'Store {i}'
        if col_name in df_aop.columns:
            store_code = row[col_name]
            if pd.notna(store_code) and store_code in kode_to_data:
                data = kode_to_data[store_code]
                kota = data['kota']
                ujp_value = data['ujp_max']

                # Tambahkan nilai UJP ke total untuk kota ini
                if kota in kota_ujp_total:
                    kota_ujp_total[kota]['total_ujp'] += ujp_value
                    kota_ujp_total[kota]['store_count'] += 1
                else:
                    kota_ujp_total[kota] = {
                        'total_ujp': ujp_value,
                        'store_count': 1
                    }

    # Jika tidak ada kota yang ditemukan
    if not kota_ujp_total:
        return "Tidak Diketahui"

    # Mencari kota dengan total UJP tertinggi
    max_kota = max(kota_ujp_total.items(), key=lambda x: x[1]['total_ujp'])

    # Menampilkan informasi debug (opsional)
    print(f"No. SPJ: {row['No. SPJ']}")
    for kota, data in kota_ujp_total.items():
        print(f"  - {kota}: {data['store_count']} store, Total UJP: {data['total_ujp']}")
    print(f"  -> Kota terpilih: {max_kota[0]} (Total UJP: {max_kota[1]['total_ujp']})")
    print()

    return max_kota[0]

# Fungsi alternatif jika hanya ingin melihat UJP per store tertinggi (bukan total)
def get_kota_by_highest_single_ujp(row):
    highest_ujp = 0
    selected_kota = "Tidak Diketahui"

    # Iterasi melalui kolom Store 1 hingga Store 50
    for i in range(1, 51):
        col_name = f'Store {i}'
        if col_name in df_aop.columns:
            store_code = row[col_name]
            if pd.notna(store_code) and store_code in kode_to_data:
                data = kode_to_data[store_code]
                ujp_value = data['ujp_max']

                # Pilih kota dengan UJP per store tertinggi
                if ujp_value > highest_ujp:
                    highest_ujp = ujp_value
                    selected_kota = data['kota']

    return selected_kota

# Pilih metode yang diinginkan:
# 1. Berdasarkan TOTAL UJP semua store di kota tersebut (rekomendasi)
print("Menggunakan metode: Kota dengan TOTAL UJP tertinggi dari semua store")
df_aop['Rute (Kota)'] = df_aop.apply(get_kota_rute_by_ujp, axis=1)

# Atau 2. Berdasarkan UJP per store tertinggi:
# print("Menggunakan metode: Kota dengan UJP per store tertinggi")
# df_aop['Rute (Kota)'] = df_aop.apply(get_kota_by_highest_single_ujp, axis=1)

# Menambahkan informasi tambahan untuk analisis
def get_ujp_info(row):
    kota_ujp_info = {}

    for i in range(1, 51):
        col_name = f'Store {i}'
        if col_name in df_aop.columns:
            store_code = row[col_name]
            if pd.notna(store_code) and store_code in kode_to_data:
                data = kode_to_data[store_code]
                kota = data['kota']
                if kota not in kota_ujp_info:
                    kota_ujp_info[kota] = {
                        'stores': [],
                        'total_ujp': 0
                    }
                kota_ujp_info[kota]['stores'].append(store_code)
                kota_ujp_info[kota]['total_ujp'] += data['ujp_max']

    # Format informasi
    info_parts = []
    for kota, data in kota_ujp_info.items():
        info_parts.append(f"{kota}: {len(data['stores'])} store, UJP={data['total_ujp']:.0f}")

    return "; ".join(info_parts) if info_parts else "Tidak ada data"

# Tambahkan kolom informasi detail
df_aop['Detail Rute & UJP'] = df_aop.apply(get_ujp_info, axis=1)

# Menampilkan preview hasil
print("\n" + "="*80)
print("PREVIEW DATA AOP DENGAN KOLOM RUTE (KOTA) BERDASARKAN UJP TERTINGGI")
print("="*80)
print("\n5 Data Pertama:")
print(df_aop[['No. SPJ', 'Nama Driver 1', 'Total Store', 'Rute (Kota)', 'Detail Rute & UJP']].head())

# Analisis statistik
print("\n" + "="*80)
print("ANALISIS STATISTIK")
print("="*80)

# Hitung distribusi kota
kota_distribution = df_aop['Rute (Kota)'].value_counts()
print("\nDistribusi Kota Berdasarkan UJP Tertinggi:")
print(kota_distribution)

# Rata-rata jumlah store per rute
avg_stores_per_rute = df_aop.groupby('Rute (Kota)')['Total Store'].mean()
print("\nRata-rata Jumlah Store per Rute Kota:")
print(avg_stores_per_rute)

# Menyimpan hasil ke file Excel baru
output_file = 'LHD_AOP_Semarang_Desember_2025_Dengan_Rute_UJP.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_aop.to_excel(writer, sheet_name='AOP', index=False)
    df_master.to_excel(writer, sheet_name='data_master', index=False)

    # Buat sheet tambahan untuk analisis
    summary_df = pd.DataFrame({
        'Kota': kota_distribution.index,
        'Jumlah Pengiriman': kota_distribution.values
    })
    summary_df.to_excel(writer, sheet_name='Analisis_Rute', index=False)

print(f"\nFile berhasil disimpan sebagai: {output_file}")
print("Sheet yang tersedia:")
print("1. AOP - Data lengkap dengan kolom Rute (Kota) dan Detail Rute & UJP")
print("2. data_master - Data master asli")
print("3. Analisis_Rute - Ringkasan distribusi kota")

# Download file
files.download(output_file)

# Tampilkan contoh perhitungan untuk baris pertama
print("\n" + "="*80)
print("CONTOH PERHITUNGAN UNTUK BARIS PERTAMA:")
print("="*80)
first_row = df_aop.iloc[0]
print(f"No. SPJ: {first_row['No. SPJ']}")
print(f"Nama Driver: {first_row['Nama Driver 1']}")
print(f"Store yang dikunjungi: {first_row['Store 1']}")
print(f"Kota: {kode_to_data.get(first_row['Store 1'], {}).get('kota', 'Tidak Diketahui')}")
print(f"Nilai UJP Max: {kode_to_data.get(first_row['Store 1'], {}).get('ujp_max', 0)}")
print(f"Rute (Kota) Terpilih: {first_row['Rute (Kota)']}")
print(f"Detail: {first_row['Detail Rute & UJP']}")

In [ ]:
# ==========================================================
# INSTALL
# ==========================================================
!pip install pandas openpyxl --quiet

# ==========================================================
# IMPORT
# ==========================================================
import pandas as pd
from google.colab import files

# ==========================================================
# UPLOAD FILE
# ==========================================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# ==========================================================
# LOAD EXCEL
# ==========================================================
excel = pd.ExcelFile(file_name)

df_lhd = pd.read_excel(excel, sheet_name='Report_LHD')
df_master = pd.read_excel(excel, sheet_name='data_master')

# Rapikan header
df_lhd.columns = df_lhd.columns.str.strip()
df_master.columns = df_master.columns.str.strip()

# ==========================================================
# 1. MEMBUAT LOOKUP TOKO -> KOTA
# ==========================================================
df_master['Kode Toko'] = df_master['Kode Toko'].astype(str).str.strip()
df_master['Kota'] = df_master['Kota'].astype(str).str.upper().str.strip()

store_to_kota = dict(zip(df_master['Kode Toko'], df_master['Kota']))

print("Total toko master terbaca:", len(store_to_kota))

# ==========================================================
# 2. DETEKSI KOLOM STORE & UJP
# ==========================================================
store_cols = sorted([c for c in df_lhd.columns if 'Store ' in c])
ujp_cols   = sorted([c for c in df_lhd.columns if 'UJP Allowance ' in c and '(Max)' not in c])

print("Jumlah store kolom :", len(store_cols))
print("Jumlah UJP kolom   :", len(ujp_cols))

# ==========================================================
# 3. HITUNG RUTE BERDASARKAN TOTAL UJP
# ==========================================================
def hitung_rute(row):

    kota_ujp = {}

    for s_col, u_col in zip(store_cols, ujp_cols):

        store = row[s_col]
        ujp   = row[u_col]

        if pd.isna(store) or pd.isna(ujp):
            continue

        store = str(store).strip()

        kota = store_to_kota.get(store)

        if kota is None:
            continue

        kota_ujp[kota] = kota_ujp.get(kota, 0) + ujp

    if not kota_ujp:
        return "Tidak Diketahui"

    return max(kota_ujp, key=kota_ujp.get)

df_lhd['Rute (Kota)'] = df_lhd.apply(hitung_rute, axis=1)

# ==========================================================
# 4. DETAIL PER KOTA
# ==========================================================
def detail_kota(row):

    kota_ujp = {}

    for s_col, u_col in zip(store_cols, ujp_cols):

        store = row[s_col]
        ujp   = row[u_col]

        if pd.isna(store) or pd.isna(ujp):
            continue

        store = str(store).strip()
        kota = store_to_kota.get(store)

        if kota is None:
            continue

        kota_ujp[kota] = kota_ujp.get(kota, 0) + ujp

    if not kota_ujp:
        return ""

    return " | ".join([f"{k}:{int(v):,}" for k,v in kota_ujp.items()])

df_lhd['Detail Kota UJP'] = df_lhd.apply(detail_kota, axis=1)

# ==========================================================
# ANALISIS
# ==========================================================
print("\nDistribusi Rute:")
print(df_lhd['Rute (Kota)'].value_counts())

# ==========================================================
# EXPORT
# ==========================================================
output = "Report_LHD_Dengan_Rute_Kota.xlsx"

with pd.ExcelWriter(output, engine='openpyxl') as writer:
    df_lhd.to_excel(writer, sheet_name='Report_LHD_Hasil', index=False)
    df_master.to_excel(writer, sheet_name='data_master', index=False)

print("\nSELESAI! FILE SIAP DI DOWNLOAD")
files.download(output)


In [ ]:
# ==========================================================
# 1. INSTALL LIBRARY
# ==========================================================
!pip install pandas openpyxl --quiet

# ==========================================================
# 2. IMPORT
# ==========================================================
import pandas as pd
import numpy as np
from google.colab import files

# ==========================================================
# 3. UPLOAD FILE
# ==========================================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# ==========================================================
# 4. LOAD SHEET
# ==========================================================
excel = pd.ExcelFile(file_name)

df_lhd = pd.read_excel(excel, sheet_name='Report_LHD')
df_aop = pd.read_excel(excel, sheet_name='AOP')

# ==========================================================
# 5. NORMALISASI HEADER
# (agar beda spasi / huruf tidak bikin script rusak)
# ==========================================================
df_lhd.columns = df_lhd.columns.str.strip()
df_aop.columns = df_aop.columns.str.strip()

# ==========================================================
# 6. MEMBUAT MASTER KOTA DARI AOP
# ==========================================================
# Cari otomatis kolom kode toko & kota
kode_col = [c for c in df_aop.columns if 'kode' in c.lower()][0]
kota_col = [c for c in df_aop.columns if 'kota' in c.lower()][0]

df_aop[kode_col] = df_aop[kode_col].astype(str).str.strip()

# dictionary: kode toko -> kota
store_to_kota = dict(zip(df_aop[kode_col], df_aop[kota_col]))

print("Total Master Toko Terbaca :", len(store_to_kota))

# ==========================================================
# 7. DETEKSI KOLOM STORE & UJP OTOMATIS
# ==========================================================
store_cols = [c for c in df_lhd.columns if 'Store' in c]
ujp_cols   = [c for c in df_lhd.columns if 'UJP' in c and 'Allowance' in c]

store_cols.sort()
ujp_cols.sort()

print("Jumlah Kolom Store :", len(store_cols))
print("Jumlah Kolom UJP   :", len(ujp_cols))

# ==========================================================
# 8. HITUNG RUTE BERDASARKAN TOTAL UJP
# ==========================================================
def hitung_rute(row):

    kota_ujp = {}

    for s_col, u_col in zip(store_cols, ujp_cols):

        store = row[s_col]
        ujp   = row[u_col]

        if pd.isna(store) or pd.isna(ujp):
            continue

        store = str(store).strip()

        # lookup kota dari AOP
        kota = store_to_kota.get(store)

        if kota is None:
            continue

        if kota not in kota_ujp:
            kota_ujp[kota] = 0

        kota_ujp[kota] += ujp

    if len(kota_ujp) == 0:
        return "Tidak Diketahui"

    # kota dengan UJP terbesar
    return max(kota_ujp, key=kota_ujp.get)

df_lhd['Rute (Kota)'] = df_lhd.apply(hitung_rute, axis=1)

# ==========================================================
# 9. DETAIL INFORMASI (OPSIONAL TAPI SANGAT BERGUNA)
# ==========================================================
def detail_rute(row):

    kota_ujp = {}

    for s_col, u_col in zip(store_cols, ujp_cols):

        store = row[s_col]
        ujp   = row[u_col]

        if pd.isna(store) or pd.isna(ujp):
            continue

        store = str(store).strip()
        kota = store_to_kota.get(store)

        if kota is None:
            continue

        kota_ujp[kota] = kota_ujp.get(kota, 0) + ujp

    if len(kota_ujp) == 0:
        return ""

    return " | ".join([f"{k} : {int(v):,}" for k,v in kota_ujp.items()])

df_lhd['Detail Kota UJP'] = df_lhd.apply(detail_rute, axis=1)

# ==========================================================
# 10. ANALISIS
# ==========================================================
print("\nDistribusi Rute:")
print(df_lhd['Rute (Kota)'].value_counts())

# ==========================================================
# 11. EXPORT
# ==========================================================
output_file = "Report_LHD_Dengan_Rute.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_lhd.to_excel(writer, sheet_name='Report_LHD_Hasil', index=False)
    df_aop.to_excel(writer, sheet_name='AOP_Master', index=False)

print("\nSELESAI - FILE SIAP DOWNLOAD")
files.download(output_file)


# Data Trip

In [ ]:
import pandas as pd

# ===============================
# LOAD FILE
# ===============================
file_path = "/content/Data Bitrans Januari-Desember.xlsx"
df = pd.read_excel(file_path, sheet_name="Data_Trip")

# ===============================
# MAPPING BULAN
# ===============================
bulan_map = {
    "Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
    "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12
}
reverse_bulan = {v:k for k,v in bulan_map.items()}

df["bulan_num"] = df["bulan"].map(bulan_map)

# ===============================
# FIRST & LAST APPEARANCE OP
# ===============================
first_month = df.groupby("lokasi")["bulan_num"].min().reset_index(name="bulan_aktif_awal")
last_month  = df.groupby("lokasi")["bulan_num"].max().reset_index(name="bulan_aktif_akhir")

df_op = first_month.merge(last_month, on="lokasi")

# ===============================
# REKAP BULANAN
# ===============================
rekap = []

for i in range(1,13):
    op_running = df[df["bulan_num"]==i]["lokasi"].nunique()
    op_baru = df_op[df_op["bulan_aktif_awal"]==i]["lokasi"].tolist()
    op_nonaktif = df_op[df_op["bulan_aktif_akhir"]==i]["lokasi"].tolist()

    rekap.append({
        "Bulan": reverse_bulan[i],
        "Jumlah_OP_Running": op_running,
        "OP_Baru_Running": ", ".join(op_baru),
        "Jumlah_OP_Baru": len(op_baru),
        "OP_Non_Aktif": ", ".join(op_nonaktif),
        "Jumlah_OP_Non_Aktif": len(op_nonaktif)
    })

rekap_df = pd.DataFrame(rekap)

# ===============================
# EXPORT
# ===============================
rekap_df.to_excel("REKAP_OP_JANUARI_DESEMBER.xlsx", index=False)
rekap_df


# Monitoring Rekon Bitrans

In [ ]:
import pandas as pd
from datetime import datetime
import gspread
from google.colab import auth, files
from google.auth import default
import io
import pytz

print("="*60)
print("GENERATE DATABASE PENGIRIMAN - PER PRODUK (FINAL)")
print("="*60)

# ================= SET TIMEZONE =================
jakarta_tz = pytz.timezone('Asia/Jakarta')
now_jakarta = datetime.now(jakarta_tz)
print(f"Waktu mulai: {now_jakarta.strftime('%Y-%m-%d %H:%M:%S')}")
print("-"*60)

# ================= UPLOAD FILE =================
print("\n[1/6] Upload file Excel / CSV")
uploaded = files.upload()
if not uploaded:
    raise SystemExit("❌ Tidak ada file diupload")

file_name = list(uploaded.keys())[0]
file_content = io.BytesIO(uploaded[file_name])


# ================= TEXT NORMALIZER =================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.strip()
    x = " ".join(x.split())   # hapus double spasi
    x = x.lower()
    return x


# ================= BACA FILE =================
print("\n[2/6] Membaca file...")
if file_name.endswith('.csv'):
    df = pd.read_csv(file_content)
else:
    df = pd.read_excel(file_content)

df = df.dropna(how='all')
df.columns = df.columns.str.strip()

print(f"✓ File: {file_name}")
print(f"✓ Total baris: {len(df)}")

# ================= RENAME OTOMATIS =================
mapping = {}
for col in df.columns:
    c = col.lower()
    if "kode" in c and "op" in c: mapping[col] = "Kode OP"
    if "nama" in c and "op" in c: mapping[col] = "Nama OP"
    if "customer" in c: mapping[col] = "Customer"
    if "produk" in c: mapping[col] = "Produk"
    if "sla" in c: mapping[col] = "Tanggal SLA"
    if "ujp" in c: mapping[col] = "UJP"
    if "charging" in c: mapping[col] = "Charging"
    if "status" in c: mapping[col] = "Status Rekon"

df = df.rename(columns=mapping)

required_cols = ['Kode OP','Nama OP','Customer','Produk','Tanggal SLA','UJP','Charging','Status Rekon']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise SystemExit(f"❌ Kolom tidak ditemukan: {missing}")

print("✓ Kolom valid")

# ================= FORMAT TANGGAL =================
df['Tanggal SLA'] = pd.to_datetime(df['Tanggal SLA'], errors='coerce')
df = df[df['Tanggal SLA'].notna()].copy()
df['Tanggal'] = df['Tanggal SLA'].dt.date

print(f"✓ Data valid tanggal: {len(df)} baris")

# ================= AUTH GOOGLE SHEET =================
print("\n[3/6] Autentikasi Google Sheet...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1d5eFnUM65lzpaJUJXSdpZG0BmK70SCf3TF5K4tljCZw/edit?usp=sharing"
sh = gc.open_by_url(SPREADSHEET_URL)

print("✓ Connected to Google Sheet")

# ================= LOAD MASTER =================
print("\n[4/6] Load Master_OP_OH...")

dict_master = {}

try:
    ws_master = sh.worksheet("Master_OP_OH")
    data_master = ws_master.get_all_values()

    if len(data_master) > 1:
        df_master = pd.DataFrame(data_master[1:], columns=data_master[0])
        df_master.columns = df_master.columns.str.strip()

        # mapping kolom WAJIB
        col_kode_op = [c for c in df_master.columns if "kode op" in c.lower()][0]
        col_nama_op = [c for c in df_master.columns if "nama op" in c.lower() and "lama" not in c.lower()][0]
        col_customer = [c for c in df_master.columns if c.lower() == "customer"][0]
        col_toh = [c for c in df_master.columns if "toh" in c.lower() or "woh" in c.lower()][0]

        df_master = df_master[[col_kode_op, col_nama_op, col_customer, col_toh]].copy()

        # CLEAN TEXT (WAJIB)
        df_master['key'] = (
            df_master[col_kode_op].apply(clean_text) + "|" +
            df_master[col_nama_op].apply(clean_text) + "|" +
            df_master[col_customer].apply(clean_text)
        )

        for _, r in df_master.iterrows():
            dict_master[r['key']] = str(r[col_toh]).strip()

        print(f"✓ Master loaded: {len(dict_master)} data (3 key validation)")

    else:
        print("⚠️ Master kosong")

except Exception as e:
    print("❌ ERROR LOAD MASTER:", e)

# ================= AGREGASI PER PRODUK =================
print("\n[5/6] Agregasi data...")

hasil = []

grouped = df.groupby(['Kode OP','Customer','Produk','Tanggal'])

for (kode_op, customer, produk, tanggal), group in grouped:

    jumlah_ritase = len(group)

    status_series = group['Status Rekon'].astype(str).str.strip().str.lower()
    jumlah_rekon = (status_series == 'rekon').sum()
    jumlah_non = jumlah_ritase - jumlah_rekon

    status_final = "Rekon" if jumlah_rekon == jumlah_ritase else "Non - Rekon"

    ujp_total = pd.to_numeric(group['UJP'], errors='coerce').fillna(0).sum()
    charging_total = pd.to_numeric(group['Charging'], errors='coerce').fillna(0).sum()

    nama_op = group['Nama OP'].iloc[0]

    key_lookup = (
    clean_text(kode_op) + "|" +
    clean_text(nama_op) + "|" +
    clean_text(customer)
)

    nama_toh_woh = dict_master.get(key_lookup, "TIDAK DITEMUKAN")
    if nama_toh_woh == "TIDAK DITEMUKAN":
        print("MASTER BELUM ADA:", kode_op, "|", nama_op, "|", customer)

    # Append to hasil for all processed groups, not just missing ones
    hasil.append({
        'Kode OP': kode_op,
        'Nama OP': nama_op,
        'Customer': customer,
        'Produk': produk,
        'Tanggal SLA': tanggal.strftime('%Y-%m-%d'),
        'UJP': ujp_total,
        'Charging': charging_total,
        'Status Rekon': status_final,
        'Nama TOH/WOH': nama_toh_woh,
        'Jumlah Non-Rekon': jumlah_non,
        'Jumlah Rekon': jumlah_rekon,
        'Jumlah Ritase': jumlah_ritase,
        'Tanggal Update': now_jakarta.strftime('%Y-%m-%d %H:%M:%S')
    })

df_new = pd.DataFrame(hasil)
print(f"✓ Hasil agregasi: {len(df_new)} baris")

# ================= DATABASE =================
headers = [
    'Kode OP','Nama OP','Customer','Produk','Tanggal SLA','UJP','Charging',
    'Status Rekon','Nama TOH/WOH','Jumlah Non-Rekon',
    'Jumlah Rekon','Jumlah Ritase','Tanggal Update'
]

try:
    ws_db = sh.worksheet("Database")
    data_existing = ws_db.get_all_values()

    if len(data_existing) > 1:
        df_existing = pd.DataFrame(data_existing[1:], columns=data_existing[0])
    else:
        df_existing = pd.DataFrame(columns=headers)

except:
    ws_db = sh.add_worksheet(title="Database", rows=2000, cols=20)
    df_existing = pd.DataFrame(columns=headers)

# ================= UPDATE LOGIC =================
df_new['Key'] = (
    df_new['Kode OP'].astype(str) + "|" +
    df_new['Customer'].astype(str) + "|" +
    df_new['Produk'].astype(str) + "|" +
    df_new['Tanggal SLA'].astype(str)
)

if not df_existing.empty:

    df_existing['Key'] = (
        df_existing['Kode OP'].astype(str) + "|" +
        df_existing['Customer'].astype(str) + "|" +
        df_existing['Produk'].astype(str) + "|" +
        df_existing['Tanggal SLA'].astype(str)
    )

    df_existing = df_existing.set_index('Key')
    df_new = df_new.set_index('Key')

    df_existing.update(df_new)

    df_final = pd.concat([
        df_existing,
        df_new[~df_new.index.isin(df_existing.index)]
    ])

    df_final.reset_index(drop=True, inplace=True)

else:
    df_final = df_new.copy()

# ================= FORMAT & SORT =================
numeric_cols = ['UJP','Charging','Jumlah Non-Rekon','Jumlah Rekon','Jumlah Ritase']
for col in numeric_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)

df_final['Tanggal SLA_dt'] = pd.to_datetime(df_final['Tanggal SLA'])
df_final = df_final.sort_values(['Kode OP','Customer','Produk','Tanggal SLA_dt'])
df_final = df_final.drop(columns=['Tanggal SLA_dt'])

# ================= UPLOAD =================
for col in headers:
    if col not in df_final.columns:
        df_final[col] = ""

data_upload = [headers] + df_final[headers].fillna('').values.tolist()

ws_db.clear()
ws_db.update(data_upload)

print(f"\n✓ {len(df_final)} baris berhasil ditulis ke Database")

# ================= STATISTIK =================
print("\nSTATISTIK:")
print(f"Total Ritase      : {df_final['Jumlah Ritase'].sum():,.0f}")
print(f"Total Rekon       : {df_final['Jumlah Rekon'].sum():,.0f}")
print(f"Total Non-Rekon   : {df_final['Jumlah Non-Rekon'].sum():,.0f}")
print(f"Total UJP         : Rp {df_final['UJP'].sum():,.0f}")
print(f"Total Charging    : Rp {df_final['Charging'].sum():,.0f}")

print("\nSELESAI ✅ FINAL VERSION")


GENERATE DATABASE PENGIRIMAN - PER PRODUK (FINAL)
Waktu mulai: 2026-03-01 01:40:15
------------------------------------------------------------

[1/6] Upload file Excel / CSV


Saving Pengiriman Harian 2026-02-01 Sampai 2026-02-08.xlsx to Pengiriman Harian 2026-02-01 Sampai 2026-02-08.xlsx

[2/6] Membaca file...
✓ File: Pengiriman Harian 2026-02-01 Sampai 2026-02-08.xlsx
✓ Total baris: 112530
✓ Kolom valid
✓ Data valid tanggal: 112530 baris

[3/6] Autentikasi Google Sheet...
✓ Connected to Google Sheet

[4/6] Load Master_OP_OH...
✓ Master loaded: 133 data (3 key validation)

[5/6] Agregasi data...
✓ Hasil agregasi: 4771 baris

✓ 4771 baris berhasil ditulis ke Database

STATISTIK:
Total Ritase      : 112,530
Total Rekon       : 100,331
Total Non-Rekon   : 12,199
Total UJP         : Rp 4,452,827,324
Total Charging    : Rp 79,679,757,714

SELESAI ✅ FINAL VERSION


# Data Belum Rekon (DEV)

In [ ]:
import pandas as pd
from datetime import datetime
import gspread
from google.colab import auth, files
from google.auth import default
import io
import pytz

# ================= UPLOAD =================
print("UPLOAD FILE EXCEL")
uploaded = files.upload()

# Add a check for uploaded files
if not uploaded:
    raise SystemExit("❌ Tidak ada file diupload!")

file_name = list(uploaded.keys())[0]
file_content = io.BytesIO(uploaded[file_name])

if file_name.endswith('.csv'):
    df = pd.read_csv(file_content)
else:
    df = pd.read_excel(file_content)

df.columns = df.columns.str.strip()

# ================= RENAME OTOMATIS =================
mapping = {}
for col in df.columns:
    c = col.lower()
    if "kode" in c and "op" in c: mapping[col] = "Kode OP"
    elif "nama" in c and "op" in c: mapping[col] = "Nama OP"
    elif "customer" in c: mapping[col] = "Customer"
    elif "spj" in c: mapping[col] = "SPJ"
    elif "produk" in c: mapping[col] = "Produk"
    # Make the 'Tanggal SLA' mapping more specific to avoid conflicts
    elif "sla" in c and "tanggal" in c: mapping[col] = "Tanggal SLA"
    elif "ujp" in c: mapping[col] = "UJP"
    elif "charging" in c: mapping[col] = "Charging"
    elif "status" in c: mapping[col] = "Status Rekon"

df = df.rename(columns=mapping)

required = ["Kode OP","Nama OP","Customer","SPJ","Produk",
            "Tanggal SLA","UJP","Charging","Status Rekon"]

for col in required:
    if col not in df.columns:
        df[col] = ""

df["Tanggal SLA"] = pd.to_datetime(df["Tanggal SLA"], errors="coerce")
df = df[df["Tanggal SLA"].notna()]
df["Tanggal SLA"] = df["Tanggal SLA"].dt.strftime("%Y-%m-%d")

df["Status Rekon"] = df["Status Rekon"].astype(str).str.lower().str.strip()

# ================= DATA TERBARU =================
df_nonrekon_baru = df[df["Status Rekon"].str.contains("non")].copy()
df_rekon_baru = df[df["Status Rekon"].str.contains("rekon") & ~df["Status Rekon"].str.contains("non")].copy()

# Buat key unik
for d in [df_nonrekon_baru, df_rekon_baru]:
    d["Key"] = (
        d["Kode OP"].astype(str) + "|" +
        d["SPJ"].astype(str) + "|" +
        d["Produk"].astype(str) + "|" +
        d["Tanggal SLA"].astype(str)
    )

# ================= AUTH SHEET =================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/12FcleafDZHI-OkyxWEfFQM4BXcx2RBNIfG2ShnoptGo/edit?usp=sharing"
sh = gc.open_by_url(SPREADSHEET_URL)

try:
    ws = sh.worksheet("Data_Rekon")
    data_existing = ws.get_all_values()
    if len(data_existing) > 1:
        df_existing = pd.DataFrame(data_existing[1:], columns=data_existing[0])
    else:
        df_existing = pd.DataFrame(columns=required)
except:
    ws = sh.add_worksheet(title="Data_Rekon", rows=1000, cols=20)
    df_existing = pd.DataFrame(columns=required)

# ================= PROCESS MERGE =================
if not df_existing.empty:
    df_existing["Key"] = (
        df_existing["Kode OP"].astype(str) + "|" +
        df_existing["SPJ"].astype(str) + "|" +
        df_existing["Produk"].astype(str) + "|" +
        df_existing["Tanggal SLA"].astype(str)
    )
else:
    # Initialize with an empty DataFrame for 'Key' column
    df_existing = pd.DataFrame(columns=required + ['Key'])

# 1️⃣ Hapus yang sekarang sudah Rekon
df_existing = df_existing[~df_existing["Key"].isin(df_rekon_baru["Key"])]

# 2️⃣ Tambahkan Non-Rekon terbaru (replace jika sudah ada)
df_existing = df_existing[~df_existing["Key"].isin(df_nonrekon_baru["Key"])]
df_final = pd.concat([df_existing, df_nonrekon_baru], ignore_index=True)

# ================= FINAL CLEAN =================
headers = required
df_final = df_final[headers]
df_final = df_final.sort_values(["Kode OP","Customer","Tanggal SLA","SPJ"])

# ================= UPDATE SHEET =================
data_upload = [headers] + df_final.fillna("").values.tolist()
ws.clear()
ws.update(data_upload)

print("SELESAI ✅")
print("Total Outstanding Saat Ini:", len(df_final))

UPLOAD FILE EXCEL


Saving Pengiriman Harian 2026-02-01 Sampai 2026-02-08.xlsx to Pengiriman Harian 2026-02-01 Sampai 2026-02-08.xlsx


/tmp/ipython-input-377/394518223.py:106: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_existing, df_nonrekon_baru], ignore_index=True)


SELESAI ✅
Total Outstanding Saat Ini: 12199


# Monitoring Rekon (DEV)

In [ ]:
import pandas as pd
from datetime import datetime
import gspread
from google.colab import auth, files
from google.auth import default
import io
import pytz

print("="*60)
print("GENERATE DATABASE PENGIRIMAN - PER PRODUK (FINAL)")
print("="*60)

# ================= SET TIMEZONE =================
jakarta_tz = pytz.timezone('Asia/Jakarta')
now_jakarta = datetime.now(jakarta_tz)
print(f"Waktu mulai: {now_jakarta.strftime('%Y-%m-%d %H:%M:%S')}")
print("-"*60)

# ================= UPLOAD FILE =================
print("\n[1/6] Upload file Excel / CSV")
uploaded = files.upload()
if not uploaded:
    raise SystemExit("❌ Tidak ada file diupload")

file_name = list(uploaded.keys())[0]
file_content = io.BytesIO(uploaded[file_name])


# ================= TEXT NORMALIZER =================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.strip()
    x = " ".join(x.split())   # hapus double spasi
    x = x.lower()
    return x


# ================= BACA FILE =================
print("\n[2/6] Membaca file...")
if file_name.endswith('.csv'):
    df = pd.read_csv(file_content)
else:
    df = pd.read_excel(file_content)

df = df.dropna(how='all')
df.columns = df.columns.str.strip()

print(f"✓ File: {file_name}")
print(f"✓ Total baris: {len(df)}")

# ================= RENAME OTOMATIS =================
mapping = {}
for col in df.columns:
    c = col.lower()
    if "kode" in c and "op" in c: mapping[col] = "Kode OP"
    if "nama" in c and "op" in c: mapping[col] = "Nama OP"
    if "customer" in c: mapping[col] = "Customer"
    if "produk" in c: mapping[col] = "Produk"
    if "sla" in c: mapping[col] = "Tanggal SLA"
    if "ujp" in c: mapping[col] = "UJP"
    if "charging" in c: mapping[col] = "Charging"
    if "status" in c: mapping[col] = "Status Rekon"

df = df.rename(columns=mapping)

required_cols = ['Kode OP','Nama OP','Customer','Produk','Tanggal SLA','UJP','Charging','Status Rekon']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise SystemExit(f"❌ Kolom tidak ditemukan: {missing}")

print("✓ Kolom valid")

# ================= FORMAT TANGGAL =================
df['Tanggal SLA'] = pd.to_datetime(df['Tanggal SLA'], errors='coerce')
df = df[df['Tanggal SLA'].notna()].copy()
df['Tanggal'] = df['Tanggal SLA'].dt.date

print(f"✓ Data valid tanggal: {len(df)} baris")

# ================= AUTH GOOGLE SHEET =================
print("\n[3/6] Autentikasi Google Sheet...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1d5eFnUM65lzpaJUJXSdpZG0BmK70SCf3TF5K4tljCZw/edit?usp=sharing"
sh = gc.open_by_url(SPREADSHEET_URL)

print("✓ Connected to Google Sheet")

# ================= LOAD MASTER =================
print("\n[4/6] Load Master_OP_OH...")

dict_master = {}

try:
    ws_master = sh.worksheet("Master_OP_OH")
    data_master = ws_master.get_all_values()

    if len(data_master) > 1:
        df_master = pd.DataFrame(data_master[1:], columns=data_master[0])
        df_master.columns = df_master.columns.str.strip()

        # mapping kolom WAJIB
        col_kode_op = [c for c in df_master.columns if "kode op" in c.lower()][0]
        col_nama_op = [c for c in df_master.columns if "nama op" in c.lower() and "lama" not in c.lower()][0]
        col_customer = [c for c in df_master.columns if c.lower() == "customer"][0]
        col_toh = [c for c in df_master.columns if "toh" in c.lower() or "woh" in c.lower()][0]

        df_master = df_master[[col_kode_op, col_nama_op, col_customer, col_toh]].copy()

        # CLEAN TEXT (WAJIB)
        df_master['key'] = (
            df_master[col_kode_op].apply(clean_text) + "|" +
            df_master[col_nama_op].apply(clean_text) + "|" +
            df_master[col_customer].apply(clean_text)
        )

        for _, r in df_master.iterrows():
            dict_master[r['key']] = str(r[col_toh]).strip()

        print(f"✓ Master loaded: {len(dict_master)} data (3 key validation)")

    else:
        print("⚠️ Master kosong")

except Exception as e:
    print("❌ ERROR LOAD MASTER:", e)

# ================= AGREGASI PER PRODUK =================
print("\n[5/6] Agregasi data...")

hasil = []

grouped = df.groupby(['Kode OP','Customer','Produk','Tanggal'])

for (kode_op, customer, produk, tanggal), group in grouped:

    jumlah_ritase = len(group)

    status_series = group['Status Rekon'].astype(str).str.strip().str.lower()
    jumlah_rekon = (status_series == 'rekon').sum()
    jumlah_non = jumlah_ritase - jumlah_rekon

    status_final = "Rekon" if jumlah_rekon == jumlah_ritase else "Non - Rekon"

    ujp_total = pd.to_numeric(group['UJP'], errors='coerce').fillna(0).sum()
    charging_total = pd.to_numeric(group['Charging'], errors='coerce').fillna(0).sum()

    nama_op = group['Nama OP'].iloc[0]

    key_lookup = (
    clean_text(kode_op) + "|" +
    clean_text(nama_op) + "|" +
    clean_text(customer)
)

    nama_toh_woh = dict_master.get(key_lookup, "TIDAK DITEMUKAN")
    if nama_toh_woh == "TIDAK DITEMUKAN":
        print("MASTER BELUM ADA:", kode_op, "|", nama_op, "|", customer)

    # Append to hasil for all processed groups, not just missing ones
    hasil.append({
        'Kode OP': kode_op,
        'Nama OP': nama_op,
        'Customer': customer,
        'Produk': produk,
        'Tanggal SLA': tanggal.strftime('%Y-%m-%d'),
        'UJP': ujp_total,
        'Charging': charging_total,
        'Status Rekon': status_final,
        'Nama TOH/WOH': nama_toh_woh,
        'Jumlah Non-Rekon': jumlah_non,
        'Jumlah Rekon': jumlah_rekon,
        'Jumlah Ritase': jumlah_ritase,
        'Tanggal Update': now_jakarta.strftime('%Y-%m-%d %H:%M:%S')
    })

df_new = pd.DataFrame(hasil)
print(f"✓ Hasil agregasi: {len(df_new)} baris")

# ================= DATABASE =================
headers = [
    'Kode OP','Nama OP','Customer','Produk','Tanggal SLA','UJP','Charging',
    'Status Rekon','Nama TOH/WOH','Jumlah Non-Rekon',
    'Jumlah Rekon','Jumlah Ritase','Tanggal Update'
]

try:
    ws_db = sh.worksheet("Database")
    data_existing = ws_db.get_all_values()

    if len(data_existing) > 1:
        df_existing = pd.DataFrame(data_existing[1:], columns=data_existing[0])
    else:
        df_existing = pd.DataFrame(columns=headers)

except:
    ws_db = sh.add_worksheet(title="Database", rows=2000, cols=20)
    df_existing = pd.DataFrame(columns=headers)

# ================= UPDATE LOGIC =================
df_new['Key'] = (
    df_new['Kode OP'].astype(str) + "|" +
    df_new['Customer'].astype(str) + "|" +
    df_new['Produk'].astype(str) + "|" +
    df_new['Tanggal SLA'].astype(str)
)

if not df_existing.empty:

    df_existing['Key'] = (
        df_existing['Kode OP'].astype(str) + "|" +
        df_existing['Customer'].astype(str) + "|" +
        df_existing['Produk'].astype(str) + "|" +
        df_existing['Tanggal SLA'].astype(str)
    )

    df_existing = df_existing.set_index('Key')
    df_new = df_new.set_index('Key')

    df_existing.update(df_new)

    df_final = pd.concat([
        df_existing,
        df_new[~df_new.index.isin(df_existing.index)]
    ])

    df_final.reset_index(drop=True, inplace=True)

else:
    df_final = df_new.copy()

# ================= FORMAT & SORT =================
numeric_cols = ['UJP','Charging','Jumlah Non-Rekon','Jumlah Rekon','Jumlah Ritase']
for col in numeric_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)

df_final['Tanggal SLA_dt'] = pd.to_datetime(df_final['Tanggal SLA'])
df_final = df_final.sort_values(['Kode OP','Customer','Produk','Tanggal SLA_dt'])
df_final = df_final.drop(columns=['Tanggal SLA_dt'])

# ================= UPLOAD =================
for col in headers:
    if col not in df_final.columns:
        df_final[col] = ""

data_upload = [headers] + df_final[headers].fillna('').values.tolist()

ws_db.clear()
ws_db.update(data_upload)

print(f"\n✓ {len(df_final)} baris berhasil ditulis ke Database")

# ================= STATISTIK =================
print("\nSTATISTIK:")
print(f"Total Ritase      : {df_final['Jumlah Ritase'].sum():,.0f}")
print(f"Total Rekon       : {df_final['Jumlah Rekon'].sum():,.0f}")
print(f"Total Non-Rekon   : {df_final['Jumlah Non-Rekon'].sum():,.0f}")
print(f"Total UJP         : Rp {df_final['UJP'].sum():,.0f}")
print(f"Total Charging    : Rp {df_final['Charging'].sum():,.0f}")

print("\nSELESAI ✅ FINAL VERSION")


# New Section

In [ ]:
# ================= GOOGLE COLAB SCRIPT =================
# Upload file, proses data rekon, update spreadsheet, lookup nama TOH/WOH,
# dan catat tanggal update.

import pandas as pd
from datetime import datetime
import gspread
from google.colab import auth, files
from google.auth import default
import io
import pytz

print("🚀 Memulai proses...")

# ================= UPLOAD FILE =================
print("📤 Silakan upload file Excel (format .xlsx atau .csv)")
uploaded = files.upload()

if not uploaded:
    raise SystemExit("❌ Tidak ada file diupload!")

file_name = list(uploaded.keys())[0]
file_content = io.BytesIO(uploaded[file_name])

if file_name.endswith('.csv'):
    df = pd.read_csv(file_content)
else:
    df = pd.read_excel(file_content)

df.columns = df.columns.str.strip()
print("✅ File berhasil dibaca.")

# ================= RENAME OTOMATIS =================
mapping = {}
for col in df.columns:
    c = col.lower()
    if "kode" in c and "op" in c:
        mapping[col] = "Kode OP"
    elif "nama" in c and "op" in c:
        mapping[col] = "Nama OP"
    elif "customer" in c:
        mapping[col] = "Customer"
    elif "spj" in c:
        mapping[col] = "SPJ"
    elif "produk" in c:
        mapping[col] = "Produk"
    elif "sla" in c and "tanggal" in c:
        mapping[col] = "Tanggal SLA"
    elif "ujp" in c:
        mapping[col] = "UJP"
    elif "charging" in c:
        mapping[col] = "Charging"
    elif "status" in c:
        mapping[col] = "Status Rekon"

df = df.rename(columns=mapping)

# Pastikan kolom wajib ada, isi kosong jika tidak
required = [
    "Kode OP", "Nama OP", "Customer", "SPJ", "Produk",
    "Tanggal SLA", "UJP", "Charging", "Status Rekon"
]
for col in required:
    if col not in df.columns:
        df[col] = ""

# Konversi Tanggal SLA
df["Tanggal SLA"] = pd.to_datetime(df["Tanggal SLA"], errors="coerce")
df = df[df["Tanggal SLA"].notna()]
df["Tanggal SLA"] = df["Tanggal SLA"].dt.strftime("%Y-%m-%d")

# Bersihkan kolom Status Rekon
df["Status Rekon"] = df["Status Rekon"].astype(str).str.lower().str.strip()

# Pisahkan Non-Rekon dan Rekon
df_nonrekon_baru = df[df["Status Rekon"].str.contains("non")].copy()
df_rekon_baru = df[df["Status Rekon"].str.contains("rekon") & ~df["Status Rekon"].str.contains("non")].copy()

# Buat key unik (untuk identifikasi baris)
for d in [df_nonrekon_baru, df_rekon_baru]:
    d["Key"] = (
        d["Kode OP"].astype(str) + "|" +
        d["SPJ"].astype(str) + "|" +
        d["Produk"].astype(str) + "|" +
        d["Tanggal SLA"].astype(str)
    )

# ================= AUTHENTIKASI GOOGLE SHEET =================
print("🔐 Mengautentikasi ke Google Sheets...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/12FcleafDZHI-OkyxWEfFQM4BXcx2RBNIfG2ShnoptGo/edit?usp=sharing"
sh = gc.open_by_url(SPREADSHEET_URL)
print("✅ Terhubung ke spreadsheet.")

# ================= AMBIL DATA EXISTING =================
try:
    ws = sh.worksheet("Data_Rekon")
    data_existing = ws.get_all_values()
    if len(data_existing) > 1:
        df_existing = pd.DataFrame(data_existing[1:], columns=data_existing[0])
    else:
        df_existing = pd.DataFrame(columns=required)
except:
    # Buat sheet baru jika belum ada
    ws = sh.add_worksheet(title="Data_Rekon", rows=1000, cols=20)
    df_existing = pd.DataFrame(columns=required)

# ================= PROSES MERGE =================
# Buat key untuk data existing
if not df_existing.empty:
    df_existing["Key"] = (
        df_existing["Kode OP"].astype(str) + "|" +
        df_existing["SPJ"].astype(str) + "|" +
        df_existing["Produk"].astype(str) + "|" +
        df_existing["Tanggal SLA"].astype(str)
    )
else:
    df_existing = pd.DataFrame(columns=required + ['Key'])

# 1️⃣ Hapus baris yang sekarang sudah Rekon (sesuai file terbaru)
df_existing = df_existing[~df_existing["Key"].isin(df_rekon_baru["Key"])]

# 2️⃣ Tambahkan Non-Rekon terbaru (ganti jika sudah ada)
df_existing = df_existing[~df_existing["Key"].isin(df_nonrekon_baru["Key"])]
df_final = pd.concat([df_existing, df_nonrekon_baru], ignore_index=True)

# ================= LOOKUP NAMA TOH/WOH dari Master_OP_OH =================
print("🔍 Melakukan lookup Nama TOH/WOH dari sheet Master_OP_OH...")
try:
    ws_master = sh.worksheet("Master_OP_OH")
    master_data = ws_master.get_all_values()
    if len(master_data) > 1:
        df_master = pd.DataFrame(master_data[1:], columns=master_data[0])
    else:
        df_master = pd.DataFrame(columns=["Kode OP", "Nama OP", "Customer", "Nama TOH/WOH"])
        print("⚠️ Sheet Master_OP_OH kosong, lookup tidak akan menghasilkan data.")
except Exception as e:
    print(f"⚠️ Tidak dapat mengakses sheet Master_OP_OH: {e}")
    df_master = pd.DataFrame(columns=["Kode OP", "Nama OP", "Customer", "Nama TOH/WOH"])

# Bersihkan spasi pada kolom master
for col in ["Kode OP", "Nama OP", "Customer"]:
    if col in df_master.columns:
        df_master[col] = df_master[col].astype(str).str.strip()
if "Nama TOH/WOH" not in df_master.columns:
    # Cari kolom yang mungkin berisi nama TOH (misal 'TOH', 'Nama TOH')
    for possible in ["TOH", "Nama TOH", "Nama WOH"]:
        if possible in df_master.columns:
            df_master.rename(columns={possible: "Nama TOH/WOH"}, inplace=True)
            break
    else:
        df_master["Nama TOH/WOH"] = ""  # kolom default kosong

# Buat dictionary lookup
lookup_dict = {}
for _, row in df_master.iterrows():
    key = (str(row.get("Kode OP", "")).strip(),
           str(row.get("Nama OP", "")).strip(),
           str(row.get("Customer", "")).strip())
    # Gunakan tuple sebagai key
    lookup_dict[key] = str(row.get("Nama TOH/WOH", "")).strip()

# Fungsi lookup
def get_nama_toh(row):
    key = (str(row.get("Kode OP", "")).strip(),
           str(row.get("Nama OP", "")).strip(),
           str(row.get("Customer", "")).strip())
    return lookup_dict.get(key, "")

# Tambahkan kolom "Nama TOH/WOH" ke df_final
df_final["Nama TOH/WOH"] = df_final.apply(get_nama_toh, axis=1)
print("✅ Lookup selesai.")

# ================= FINAL CLEAN & URUTKAN =================
# Susun headers dengan kolom baru
headers_baru = required + ["Nama TOH/WOH", "Status Adjustment"]
# Pastikan kolom yang diperlukan ada di df_final
for col in headers_baru:
    if col not in df_final.columns:
        df_final[col] = ""

df_final = df_final[headers_baru]
df_final = df_final.sort_values(["Kode OP", "Customer", "Tanggal SLA", "SPJ"])

# ================= UPDATE SHEET Data_Rekon =================
print("💾 Menulis ke sheet Data_Rekon...")
data_upload = [headers_baru] + df_final.fillna("").values.tolist()
ws.clear()
ws.update(data_upload)
print(f"✅ Data_Rekon diperbarui. Total baris: {len(df_final)}")

# ================= TULIS TANGGAL UPDATE KE SHEET CONFIG =================
print("📅 Mencatat waktu update ke sheet Config...")
try:
    ws_config = sh.worksheet("Config")
except:
    ws_config = sh.add_worksheet(title="Config", rows=1, cols=1)

# Waktu sekarang dalam WIB
wib = pytz.timezone('Asia/Jakarta')
now_wib = datetime.now(wib).strftime("%Y-%m-%d %H:%M:%S")
ws_config.update('A1', [[now_wib]])
print(f"✅ Config diperbarui: {now_wib}")

print("🎉 SEMUA PROSES SELESAI!")

# ================= UNTUK LOOKUP STATUS PENYESUAIAN DATA =================
# ================= LOOKUP STATUS ADJUSTMENT =================
print("🔍 Lookup Status Adjustment dari DATA_REKAP...")

try:
    # ID Spreadsheet
    ID_REKAP = "1y1iFq7BI3mDiNFpGTl2uQLCFdwexNYu8NfVLsVSVChY"
    ID_REKON = "12FcleafDZHI-OkyxWEfFQM4BXcx2RBNIfG2ShnoptGo"

    # Buka Sheet DATA_REKAP
    sheet_rekap = gc.open_by_key(ID_REKAP).worksheet("DATA_REKAP")
    data_rekap = sheet_rekap.get_all_values()

    if len(data_rekap) > 1:
        df_rekap = pd.DataFrame(data_rekap[1:], columns=data_rekap[0])

        # Kolom fix berdasarkan posisi
        spj_column = df_rekap.columns[6]      # Kolom G
        status_column = df_rekap.columns[15]  # Kolom P

        # Bersihkan data
        df_rekap[spj_column] = (
            df_rekap[spj_column]
            .astype(str)
            .str.strip()
            .str.replace(".0", "", regex=False)
            .str.upper()
        )

        df_rekap[status_column] = (
            df_rekap[status_column]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        # Buat mapping dictionary
        rekap_map = dict(zip(
            df_rekap[spj_column],
            df_rekap[status_column]
        ))

        print(f"✅ Total SPJ termapping: {len(rekap_map)}")

    else:
        print("⚠️ DATA_REKAP kosong")
        rekap_map = {}

except Exception as e:
    print(f"❌ Error membaca DATA_REKAP: {e}")
    rekap_map = {}

# ================= FUNGSI MAPPING =================
def map_status_adjustment(spj):
    spj = (
        str(spj)
        .strip()
        .replace(".0", "")
        .upper()
    )

    status = rekap_map.get(spj, "")

    if status == "DONE":
        return "Done"
    elif status in ["PROCESS", "PROCESSING"]:
        return "Process"
    else:
        return "No Adjustment"

# ================= BACA DATA_REKON =================
print("\n📥 Membaca Data_Rekon...")

sheet_rekon = gc.open_by_key(ID_REKON).worksheet("Data_Rekon")
data_rekon = sheet_rekon.get_all_values()

df_final = pd.DataFrame(data_rekon[1:], columns=data_rekon[0])

# Pastikan kolom SPJ ada
if "SPJ" not in df_final.columns:
    raise Exception("❌ Kolom 'SPJ' tidak ditemukan di Data_Rekon")

# Bersihkan SPJ di Data_Rekon
df_final["SPJ"] = (
    df_final["SPJ"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .str.upper()
)

# ================= APPLY MAPPING =================
print("🔄 Memproses Status Adjustment...")

df_final["Status Adjustment"] = df_final["SPJ"].apply(map_status_adjustment)

print("✅ Mapping selesai")
print("\n📊 Ringkasan hasil:")
print(df_final["Status Adjustment"].value_counts())

# ================= UPDATE SHEET DATA_REKON =================
print("\n📤 Mengupdate Sheet Data_Rekon...")

sheet_rekon.clear()

sheet_rekon.update(
    [df_final.columns.values.tolist()] +
    df_final.values.tolist()
)

print("🎉 SELESAI! Status Adjustment berhasil ditambahkan ke Data_Rekon.")

🚀 Memulai proses...
📤 Silakan upload file Excel (format .xlsx atau .csv)


Saving Book1.xlsx to Book1 (4).xlsx
✅ File berhasil dibaca.
🔐 Mengautentikasi ke Google Sheets...
✅ Terhubung ke spreadsheet.
🔍 Melakukan lookup Nama TOH/WOH dari sheet Master_OP_OH...
✅ Lookup selesai.
💾 Menulis ke sheet Data_Rekon...
✅ Data_Rekon diperbarui. Total baris: 12190
📅 Mencatat waktu update ke sheet Config...


/tmp/ipython-input-364/2497188319.py:205: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws_config.update('A1', [[now_wib]])


✅ Config diperbarui: 2026-03-01 15:53:58
🎉 SEMUA PROSES SELESAI!
🔍 Lookup Status Adjustment dari DATA_REKAP...
✅ Total SPJ termapping: 6624

📥 Membaca Data_Rekon...
🔄 Memproses Status Adjustment...
✅ Mapping selesai

📊 Ringkasan hasil:
Status Adjustment
No Adjustment    11983
Done               198
Process              9
Name: count, dtype: int64

📤 Mengupdate Sheet Data_Rekon...
🎉 SELESAI! Status Adjustment berhasil ditambahkan ke Data_Rekon.
